# Set up a regional MOM6 run within CESM framework (with CrocoDash grid generation)

There are three main sections: 
1. First we create the MOM6 experiment using the [regional-mom6](https://github.com/COSIMA/regional-mom6/) package. This puts everything we need into two repositories, referred to as the `mom_run_dir` and `mom_input_dir`. These contain all of the configuration files (`MOM_input` etc.) and netcdf input files (`hgrid.nc`, OBC segments etc.) respectively. An `experiment` object is also created within the notebook, containing all the information we need about the regional mom6 setup to pass onto CESM.

2. A new MOM6 CESM case is created in the usual way by cloning the CESM repo and running the `create_case` command. 

3. We modify the new CESM case to work with the regional mom6 configuration we prepared earlier. We pass the path to this CESM run and the experiment object to the `setup_cesm` function, which carries out all the required modifications 

This is very much a work in progress! The goal in the short term was to get something working and make it nice later

# SECTION 1: Setup up you MOM6 regional experiment

This follows the normal workflow, copied from the `reanalysis_forced.ipynb` demo of [regional-mom6](https://github.com/COSIMA/regional-mom6/) but modified for a domain around Hawaii. See the documentation of the package for details and documentation

In [1]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path
import sys
import CrocoDash as crr
import regional_mom6 as rmom6
import CrocoDash as cd
import regional_mom6 as rmom6

In [ ]:
# For more detailed output
# import logging
# logging.basicConfig(level=logging.INFO) 
# Currently, only the regional_mom6 module reports logging information to the info level



## Step 2: Make experiment object
The `regional_mom6.experiment` contains the regional domain basics, and also generates the horizontal and vertical grids, `hgrid` and `vgrid` respectively, and sets up the directory structures. 

In [2]:
expt = rmom6.create_experiment_from_config("/glade/u/home/manishrv/documents/nwa12_0.1/mom_run/gfdl_copy.3.n3b/config.json", create_hgrid_and_vgrid = False)
glorys_path = os.path.join("/","glade","derecho","scratch","manishrv","gfdl_copy.1.n3b_inputs","glorys_large" )


regional_mom6.regional_mom6.create_experiment_from_config:INFO:Reading from config file....
regional_mom6.regional_mom6.create_experiment_from_config:INFO:Creating Empty Experiment Object....
regional_mom6.regional_mom6.create_experiment_from_config:INFO:Setting Default Variables.....
regional_mom6.regional_mom6.create_experiment_from_config:INFO:Skipping hgrid and vgrid creation....
regional_mom6.regional_mom6.create_experiment_from_config:INFO:Done!


In [4]:
# Define a mapping from the GLORYS variables and dimensions to the MOM6 ones
ocean_varnames = {"time": "time",
                  "yh": "latitude",
                  "xh": "longitude",
                  "zl": "depth",
                  "eta": "zos",
                  "u": "uo",
                  "v": "vo",
                  "tracers": {"salt": "so", "temp": "thetao"}
                  }

# Set up the initial condition
expt.setup_initial_condition(
    Path(glorys_path) / "ic_unprocessed.nc", # directory where the unprocessed initial condition is stored, as defined earlier
    ocean_varnames,
    arakawa_grid="A"
    )    

# Set up the four boundary conditions. Remember that in the glorys_path, we have four boundary files names north_unprocessed.nc etc. 
expt.setup_ocean_state_boundaries(
        Path(glorys_path),
        ocean_varnames,
        arakawa_grid = "A"
        )

INITIAL CONDITIONS
Regridding Velocities... Done.
Regridding Tracers... Done.
Regridding Free surface... Done.
Saving outputs... done setting up initial condition.
NOTE: the 'setup_run_directories' method does understand the less than four boundaries but be careful. Please check the MOM_input/override file carefully to reflect the number of boundaries you have, and their orientations. You should be able to find the relevant section in the MOM_input/override file by searching for 'segment_'. Ensure that the segment names match those in your inputdir/forcing folder
Processing south boundary...2025-01-02 07:05:22,644 - CrocoDash.rm6.regional_mom6.regridding.coords - INFO - Creating coordinates of the boundary q/u/v points
2025-01-02 07:05:22,647 - CrocoDash.rm6.regional_mom6.regridding.create_regridder - INFO - Creating Regridder
2025-01-02 07:09:54,834 - CrocoDash.rm6.regional_mom6.regridding.fill_missing_data - INFO - Filling in missing data horizontally, then vertically
2025-01-02 07:0

In [10]:
expt.setup_boundary_tides(Path("/glade/u/home/manishrv/manish_scratch_symlink/inputs_rm6/tidal_data/h_tpxo9.v1.nc"),Path("/glade/u/home/manishrv/manish_scratch_symlink/inputs_rm6/tidal_data/u_tpxo9.v1.nc"))

Processing south boundary...2024-12-30 08:03:21,403 - CrocoDash.rm6.regional_mom6.regridding.coords - INFO - Creating coordinates of the boundary q/u/v points
2024-12-30 08:03:21,406 - CrocoDash.rm6.regional_mom6.regridding.create_regridder - INFO - Creating Regridder
2024-12-30 08:03:41,862 - CrocoDash.rm6.regional_mom6.regridding.fill_missing_data - INFO - Forward filling in missing data along z-dim
2024-12-30 08:03:41,865 - CrocoDash.rm6.regional_mom6.regridding.fill_missing_data - INFO - Forward filling in missing data along z-dim
2024-12-30 08:03:41,886 - CrocoDash.rm6.regional_mom6.regridding.add_or_update_time_dim - INFO - Adding time dimension
2024-12-30 08:03:41,894 - CrocoDash.rm6.regional_mom6.regridding.coords - INFO - Creating coordinates of the boundary q/u/v points
2024-12-30 08:03:41,896 - CrocoDash.rm6.regional_mom6.regridding.add_secondary_dimension - INFO - Adding perpendicular dimension to zamp_segment_001
2024-12-30 08:03:41,899 - CrocoDash.rm6.regional_mom6.regrid

## Step 6: Modify the default input directory to make a (hopefully) runnable configuration out of the box

This step copies the default directory and modifies the `MOM_layout` files to match your experiment by inserting the right number of x, y points and CPU layout.

To run MOM6 using the [payu infrastructure](https://github.com/payu-org/payu), provide the keyword argument `using_payu = True` to the `setup_run_directory` method and an example `config.yaml` file will be appear in the run directory. The `config.yaml` file needs to be modified manually to add the locations of executables, etc.

In [12]:
expt.setup_run_directory(surface_forcing = "jra", with_tides = True)
expt.change_MOM_parameter("MAXIMUM_DEPTH",'7000', override = True) # Fred's Vertical Coordinate has a depth of 6500

Could not find premade run directories at  /glade/u/home/manishrv/documents/nwa12_0.1/regional_mom_workflows/CrocoDash/CrocoDash/rm6/regional_mom6/demos/premade_run_directories
Perhaps the package was imported directly rather than installed with conda. Checking if this is the case... 
Found run files. Continuing...
At least one of these is requiret to set up the experiment if you're running MOM6 standalone with the FMS coupler. 
If you're running within CESM, ignore this message.
Deleting indexed OBC keys from MOM_input_dict in case we have a different number of segments


# SECTION 2: Create a blank CESM run

So far I've used Alper's [GUI](https://github.com/ESMCI/visualCaseGen?tab=readme-ov-file) branch of CESM. Clone respective branch of CESM and then run the generate case command. Below is the command I used to generate a global, MOM6 only run forced with JRA data atmosphere

`/glade/u/home/manishrv/manish_scratch_symlink/installs/clean_cesm3_0_alpha03a/cime/scripts/create_newcase --compset 1850_DATM%JRA_SLND_SICE_MOM6_SROF_SGLC_SWAV --res TL319_t232 --case /glade/u/home/manishrv/cases/gfdl_copy.3.n3b --machine derecho --run-unsupported --project p93300012 --non-local`

In [11]:
# Save the path where the new CESM config lives
CESM_path = Path(f"/glade/u/home/manishrv/cases/gfdl_copy.3.n3b")

# SECTION 3: Modify the CESM run to make it regional

In [13]:
## Regional CaseGen is part of CrocoDash

reg_cas_obj = cd.regional_casegen.cesm_tools.RegionalCaseGen()


reg_cas_obj.setup_cesm(expt, CESM_path, project = "P93300012")
## OR you can explicitly state what variables are needed
# reg_cas_obj.setup_cesm_explicit(hgrid = expt.hgrid, CESMPath = CESM_path,project = "p93300612", mom_input_dir=expt.mom_input_dir, mom_run_dir=expt.mom_run_dir,date_range = expt.date_range)

2025-01-02 14:36:27,820 - CrocoDash.regional_casegen.cesm_tools - INFO - Copying input.nml, diag_table, MOM_input_and MOM_override to /glade/u/home/manishrv/cases/gfdl_copy.3.n3b/SourceMods/src.mom


2025-01-02 14:36:27,831 - CrocoDash.regional_casegen.cesm_tools - INFO - Adding NIGLOBAL = 740, NJGLOBAL = 780, and INPUTDIR = /glade/u/home/manishrv/documents/nwa12_0.1/mom_input/gfdl_copy.3.n3b to MOM_override
2025-01-02 14:36:27,832 - CrocoDash.regional_casegen.cesm_tools - INFO - Add MOM_override to parameter_filename in input.nml
2025-01-02 14:36:27,832 - CrocoDash.regional_casegen.cesm_tools - INFO - Removing reference to MOM_layout in input.nml and add MOM_override to parameter_filename in input.nml
2025-01-02 14:36:27,833 - CrocoDash.regional_casegen.cesm_tools - INFO - 
2025-01-02 14:36:27,834 - CrocoDash.regional_casegen.cesm_tools - INFO - Move all of the forcing files out of the forcing directory to the main inputdir
2025-01-02 14:36:27,835 - CrocoDash.regional_casegen.cesm_tools - INFO - Find and replace instances of forcing/ with nothing in the MOM_input file
2025-01-02 14:36:27,836 - CrocoDash.regional_casegen.cesm_tools - INFO - Make ESMF grid and save to inputdir
2025-

Now hopefully doing the usual 

`./case.setup && ./case.build && ./case.submit` 

should at least run. Of course from here you'll have a lot of other things to fiddle around with to make it run *well!* 